# 12. Text-to-LoRA

The goal is a simple API: describe a function in English, compile it, and get back a Python callable — no internet, no external service, no fine-tuning loop at call time.

```python
fn = compile("Classify if email is urgent")
fn("ASAP please review this invoice")   # 1
fn("Team lunch is scheduled for Friday") # 0
```

Under the hood, `compile` runs a single forward pass through a learned *compiler* network that encodes the spec and decodes it into a LoRA adapter. That adapter is injected into a frozen *interpreter* model, and the whole thing is wrapped as a regular Python function. Agents can treat it like any other tool — no gradient descent, no network call, no per-invocation cost beyond local inference.

In this notebook we build this idea from scratch. We start with LoRA itself — deriving the low-rank decomposition, counting parameters, and verifying correctness on a toy regression. We then construct a hypernetwork that outputs weights for a target network and train it end-to-end. From there we formalize the text-to-LoRA training objective, implement a full toy pipeline on three synthetic classification tasks, wrap the result as a callable Python function, and finally attach the compiler to a frozen GPT-2 to show the architecture at realistic scale.

## Section 1: Motivation

**The target API.** The end product of a text-to-LoRA system is a `compile` function that converts an English spec into a callable Python function:

```python
urgent = compile("Classify if email is urgent")
billing = compile("Classify if email is about billing")

urgent("ASAP please sign this")         # 1
billing("Your invoice #1234 is due")    # 1
urgent("Team lunch on Friday")          # 0
```

Each compiled function runs entirely locally — one forward pass through a small interpreter model. No fine-tuning, no internet, no per-call fees. An agent can treat these as ordinary Python callables and attach them as tools.

**How it works.** A natural language program specification is encoded with a fine-tuned transformer (the *compiler*) and decoded into a LoRA adapter $\Delta W = BA$. That adapter is injected into a frozen *interpreter* model, which then executes the described program on incoming inputs. The base model $W_0$ is never touched — it is shared across all adapters. Task switching is $O(1)$ in the number of compilation steps.

**The key question.** How can a neural network output the *weights* of another neural network? Ordinary networks output predictions — scalars, vectors, probability distributions. Outputting weights is just a special case: the output head is widened to match the parameter count of the target layer, and the targets during training are defined implicitly by task performance rather than by a supervision signal on the weights themselves.

**Roadmap.**

1. Build LoRA from scratch, verify it on a toy linear regression.
2. Build a minimal hypernetwork that maps task IDs to weight matrices.
3. Formalize the text-to-LoRA training objective and gradient flow.
4. Implement a full two-phase toy pipeline (3 synthetic tasks, TinyTransformer interpreter).
5. Wrap the compiled adapter as a callable Python function — the agent-ready API.
6. Attach the compiler to frozen GPT-2 and time a compilation.

## Section 2: LoRA from Scratch

### 2.1 Derivation

Given a pretrained weight matrix $W_0 \in \mathbb{R}^{d \times k}$, standard fine-tuning updates every entry. LoRA instead constrains the update $\Delta W$ to be low-rank:

$$W' = W_0 + \Delta W = W_0 + BA,$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ with $r \ll \min(d, k)$. The forward pass becomes

$$h = W' x = W_0 x + B(Ax),$$

and only $B$ and $A$ are trained. Initialization follows the original paper: $A \sim \mathcal{N}(0, \sigma^2)$ and $B = 0$, so $\Delta W = BA = 0$ at the start of training. This ensures that adapter training begins from the same representation as the pretrained model.

A scaling factor $s = \alpha / r$ is applied to $\Delta W$, where $\alpha$ is a fixed constant (often equal to $r$). The effective update is $s \cdot BA$.

### 2.2 `LoRALinear` module

**Model.** Implementing the LoRA layer:

In [ ]:
import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)  # <1>
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))         # <2>
        self.scale = alpha / rank                                            # <3>
        self.linear.weight.requires_grad = False                            # <4>

    def forward(self, x, lora_A=None, lora_B=None):
        A = lora_A if lora_A is not None else self.lora_A   # <5>
        B = lora_B if lora_B is not None else self.lora_B
        return self.linear(x) + (x @ A.T @ B.T) * self.scale

1. Small-random init for $A$ so the initial $\Delta W$ is near zero.
2. Zero init for $B$ so $\Delta W = BA = 0$ exactly at initialization.
3. Scaling factor $s = \alpha / r$ applied to the LoRA delta.
4. Freeze the pretrained base weight $W_0$ — it receives no gradient.
5. `lora_A` / `lora_B` are optional overrides. When `None`, the module's own trained parameters are used (Phase 1). When provided as slices from a compiler output tensor, the computation graph flows back to the compiler (Phase 2).

### 2.3 Parameter count

For GPT-2's attention projection ($768 \to 768$), the full weight matrix has $768^2 = 589{,}824$ parameters. A LoRA adapter at rank $r$ requires $2 \times r \times 768$ parameters. At rank 4 that is $6{,}144$ — a $96\times$ reduction. We compute this ratio across a range of ranks:

In [ ]:
d = 768
full_params = d * d
print(f"{'Rank':>6}  {'LoRA params':>12}  {'Reduction':>10}")
print("-" * 34)
for rank in [1, 2, 4, 8, 16, 32, 64]:
    lora_params = 2 * rank * d
    reduction = full_params / lora_params
    print(f"{rank:>6}  {lora_params:>12,}  {reduction:>9.1f}x")

### 2.4 Toy verification

**Model.** We verify that LoRA can recover a target weight matrix $W^* \in \mathbb{R}^{8 \times 8}$ when the base weight is fixed at $W_0 = 0$. Training minimizes the MSE between $\hat{y} = \Delta W x$ and $y = W^* x$ over random inputs $x$. With $W_0 = 0$ the LoRA update $\Delta W = BA$ must itself approximate $W^*$.

In [ ]:
torch.manual_seed(42)

d = 8
W_star = torch.randn(d, d)

layer = LoRALinear(d, d, rank=4, alpha=4.0)
nn.init.zeros_(layer.linear.weight)  # fix W_0 = 0

optimizer = torch.optim.Adam(
    [layer.lora_A, layer.lora_B], lr=1e-2
)

losses = []
for step in range(2000):
    x = torch.randn(32, d)
    y_hat = layer(x)
    y_target = x @ W_star.T
    loss = (y_hat - y_target).pow(2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.6f}")

Plotting the loss curve:

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(losses, linewidth=1.5)
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("MSE loss")
plt.title("LoRA toy regression")
plt.grid(alpha=0.4, linestyle="--")
plt.tight_layout()
plt.show()

**Figure.** MSE loss (log scale) over training steps. The LoRA adapter converges to $W^*$ within 2,000 steps, confirming that a rank-4 update suffices for an $8 \times 8$ target when the rank equals the matrix size in the most compressed sense.

## Section 3: Hypernetworks

### 3.1 Concept

A **hypernetwork** $h_\theta(z) \to \phi$ takes a task descriptor $z$ and outputs the parameters $\phi$ of a *target* network $f_\phi$. The target network is then applied to the task input $x$ in the usual way: $\hat{y} = f_\phi(x)$. Training is end-to-end — gradients flow from the task loss $\ell(\hat{y}, y)$ through $f_\phi$ into $h_\theta$, so the hypernetwork learns to synthesize parameters that perform well on each task.

The key difference from multi-task learning is that $\phi$ is not stored in a lookup table indexed by task — it is *computed* by $h_\theta$ from $z$. At test time, a new task descriptor $z'$ can produce a new $\phi'$ without any retraining.

### 3.2 Toy hypernetwork

**Data.** We define 8 synthetic linear tasks. Each task $i$ has a random target matrix $W^*_i \in \mathbb{R}^{4 \times 4}$. The task descriptor is a one-hot vector $e_i \in \mathbb{R}^8$. The hypernetwork is a 2-layer MLP that maps $e_i$ to 16 numbers, reshaped into a $4 \times 4$ predicted weight matrix. Training minimizes the MSE between $h_\theta(e_i) \cdot x$ and $W^*_i \cdot x$ over random $x$.

In [ ]:
torch.manual_seed(42)

n_tasks = 8
target_weights = [torch.randn(4, 4) for _ in range(n_tasks)]


class HyperNetwork(nn.Module):
    def __init__(self, n_tasks, target_dim):
        super().__init__()
        self.n_tasks = n_tasks
        self.net = nn.Sequential(
            nn.Linear(n_tasks, 64),
            nn.ReLU(),
            nn.Linear(64, target_dim),
        )

    def forward(self, task_id):
        one_hot = torch.zeros(self.n_tasks)    # <1>
        one_hot[task_id] = 1.0
        return self.net(one_hot).view(4, 4)


hyper = HyperNetwork(n_tasks=8, target_dim=16)
optimizer = torch.optim.Adam(hyper.parameters(), lr=1e-3)

for step in range(2000):
    task_id = torch.randint(0, n_tasks, ()).item()
    W_pred = hyper(task_id)
    x = torch.randn(4, 10)
    loss = ((W_pred @ x) - (target_weights[task_id] @ x)).pow(2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (step + 1) % 500 == 0:
        print(f"Step {step+1:>5} | loss: {loss.item():.6f}")

1. One-hot encoding of the task: the hypernetwork receives a dense $\mathbb{R}^8$ vector with a single 1.0 at position `task_id`.

Verifying the recovered weights against ground truth for each task:

In [ ]:
import torch.nn.functional as F

print(f"{'Task':>6}  {'Cosine sim':>12}  {'MSE':>10}")
print("-" * 34)
with torch.no_grad():
    for i in range(n_tasks):
        W_pred = hyper(i)
        W_true = target_weights[i]
        cos = F.cosine_similarity(
            W_pred.flatten().unsqueeze(0),
            W_true.flatten().unsqueeze(0)
        ).item()
        mse = (W_pred - W_true).pow(2).mean().item()
        print(f"{i:>6}  {cos:>12.4f}  {mse:>10.6f}")

The cosine similarities are close to 1.0, confirming that the hypernetwork has memorized a distinct weight matrix for each task. Training is end-to-end: the hypernetwork never sees $W^*_i$ directly — it only sees task performance signals.

## Section 4: Text-to-LoRA Training Objective

### 4.1 Formalization

Let $\text{spec}$ be a natural language task description. The text-to-LoRA training objective is:

$$\mathcal{L}(\theta) = \mathbb{E}_{(\text{spec},\, x,\, y)} \Bigl[\ell\bigl(f_{W_0 + h_\theta(\text{enc}(\text{spec}))}(x),\; y\bigr)\Bigr],$$

where the components are:

- $\text{enc}$: a spec encoder that maps the string to a fixed-size embedding (bag-of-words, mean-pooled transformer, etc.).
- $h_\theta$: the **compiler** network, parameterized by $\theta$, that maps the spec embedding to a flat vector of LoRA parameters $(B, A)$.
- $W_0$: the frozen **interpreter** base weights — never updated during compiler training.
- $f_{W_0 + \Delta W}$: the interpreter with LoRA applied, where $\Delta W = BA$.
- $\ell$: the task loss (cross-entropy for classification, MSE for regression, etc.).

:::{.callout-important}
$W_0$ is **never modified**. The compiler outputs $\Delta W = BA$; the base model is shared across all adapters simultaneously. Merging $\Delta W$ into $W_0$ would require a separate copy of the base model per adapter, destroying the shared-weights property.

:::

### 4.2 Gradient flow

The computation graph at training time is:

```
spec → enc → h_θ → (B, A) → ΔW = BA
                                  ↓
                    x → [W₀ + ΔW] → ŷ → loss
                                           ↓
                    ←←←←← ∂loss/∂θ ←←←←←←
```

Gradients propagate from the loss through $f$ (the interpreter forward pass), through the LoRA delta $\Delta W = BA$, and back into the compiler parameters $\theta$. The interpreter's base weights $W_0$ receive no gradient — only $\theta$ (the compiler's parameters) are updated by the optimizer.

### 4.3 Two-phase training

Training a text-to-LoRA system from scratch in a single phase is unstable because the compiler is trying to hit a moving target: the interpreter is also changing. The standard approach is a two-phase curriculum.

**Phase 1 — pre-train the interpreter.** Train the interpreter with random LoRA initialization on all tasks jointly (or with task-conditioned random adapters). The goal is a stable interpreter that performs reasonably well across the task space. At the end of phase 1, the interpreter base weights $W_0$ are frozen permanently.

**Phase 2 — train the compiler.** The interpreter is now fixed. The compiler is trained end-to-end: for each (spec, $x$, $y$) triplet, the compiler generates LoRA parameters, they are injected into the frozen interpreter, and the task loss is backpropagated into the compiler only. Because the interpreter no longer changes, the compiler is optimizing against a stationary function, which stabilizes training considerably.

## Section 5: Build Our Own (Toy)

We construct three binary classification tasks over bag-of-words (BOW) features:

- **Task 0 — urgency**: classify if text contains urgency keywords (`urgent`, `asap`, `immediately`).
- **Task 1 — billing**: classify if text contains billing keywords (`invoice`, `payment`, `charge`).
- **Task 2 — meeting**: classify if text contains meeting keywords (`schedule`, `calendar`, `meeting`).

The vocabulary has 20 words; each sample is a binary BOW vector of length 20.

### 5.1 Data generation

**Data.** We define a vocabulary and a sample generator for each task:

In [ ]:
import random
random.seed(42)

VOCAB = [
    "urgent", "asap", "immediately", "invoice", "payment", "charge",
    "schedule", "calendar", "meeting", "please", "hello", "team",
    "update", "review", "note", "request", "confirm", "reply", "send", "check"
]

TASK_KEYWORDS = {
    0: {"urgent", "asap", "immediately"},
    1: {"invoice", "payment", "charge"},
    2: {"schedule", "calendar", "meeting"},
}

TASK_SPECS = {
    0: "urgency",
    1: "billing",
    2: "meeting",
}


def make_sample(task_id, positive):
    keywords = list(TASK_KEYWORDS[task_id])
    words = random.choices(VOCAB, k=4)
    if positive:
        words[0] = random.choice(keywords)   # <1>
    else:
        neutral = [w for w in VOCAB if w not in TASK_KEYWORDS[task_id]]
        words = random.choices(neutral, k=4)
    bow = torch.zeros(len(VOCAB))
    for w in words:
        if w in VOCAB:
            bow[VOCAB.index(w)] = 1.0        # <2>
    return bow, torch.tensor(1 if positive else 0, dtype=torch.long)

1. Positive samples are guaranteed to contain at least one task keyword at position 0.
2. Binary BOW encoding: 1.0 if word is present, 0.0 otherwise.

Sanity check — sample a few instances from each task:

In [ ]:
for task_id in range(3):
    x_pos, y_pos = make_sample(task_id, positive=True)
    x_neg, y_neg = make_sample(task_id, positive=False)
    pos_words = [VOCAB[i] for i, v in enumerate(x_pos) if v > 0]
    neg_words = [VOCAB[i] for i, v in enumerate(x_neg) if v > 0]
    print(f"Task {task_id} ({TASK_SPECS[task_id]})")
    print(f"  positive (label={y_pos.item()}): {pos_words}")
    print(f"  negative (label={y_neg.item()}): {neg_words}")

### 5.2 TinyTransformer interpreter

**Model.** We define a small 2-layer transformer with `LoRALinear` in the Q and V projections of each attention block. The base weights of `LoRALinear` are frozen; only the LoRA matrices are updated during phase 1, and are *set externally* by the compiler during phase 2.

In [ ]:
class SelfAttentionLoRA(nn.Module):
    def __init__(self, d_model, n_heads, rank=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = LoRALinear(d_model, d_model, rank=rank, alpha=float(rank))
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = LoRALinear(d_model, d_model, rank=rank, alpha=float(rank))
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj.weight.requires_grad = False
        self.out_proj.weight.requires_grad = False

    def forward(self, x, lora_q_A=None, lora_q_B=None, lora_v_A=None, lora_v_B=None):
        B, T, D = x.shape
        Q = self.q_proj(x, lora_A=lora_q_A, lora_B=lora_q_B).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        V = self.v_proj(x, lora_A=lora_v_A, lora_B=lora_v_B).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / self.d_head ** 0.5
        attn = scores.softmax(dim=-1)
        out = (attn @ V).transpose(1, 2).reshape(B, T, D)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, rank=4):
        super().__init__()
        self.attn = SelfAttentionLoRA(d_model, n_heads, rank)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Linear(d_model * 2, d_model),
        )

    def forward(self, x, lora_q_A=None, lora_q_B=None, lora_v_A=None, lora_v_B=None):
        x = x + self.attn(self.norm1(x), lora_q_A=lora_q_A, lora_q_B=lora_q_B,
                           lora_v_A=lora_v_A, lora_v_B=lora_v_B)
        x = x + self.ff(self.norm2(x))
        return x


class TinyTransformer(nn.Module):
    """2-layer transformer with LoRA in Q and V projections."""
    def __init__(self, vocab_size=20, d_model=64, n_heads=4, rank=4):
        super().__init__()
        self.embed = nn.Linear(vocab_size, d_model, bias=False)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, rank) for _ in range(2)
        ])
        self.head = nn.Linear(d_model, 2)
        self.rank = rank
        self.d_model = d_model
        self.embed.weight.requires_grad = False
        self.head.weight.requires_grad = False
        if self.head.bias is not None:
            self.head.bias.requires_grad = False

    def _slice_lora(self, lora_flat):   # <1>
        """Slice a flat compiler output into (A, B) pairs per (layer, projection)."""
        per_mat = self.rank * self.d_model
        idx = 0
        pairs = []
        for _ in self.layers:
            for _ in range(2):  # Q then V
                A = lora_flat[idx: idx + per_mat].view(self.rank, self.d_model)
                idx += per_mat
                B = lora_flat[idx: idx + per_mat].view(self.d_model, self.rank)
                idx += per_mat
                pairs.append((A, B))
        return pairs

    def forward(self, x, lora_flat=None):   # <2>
        x = self.embed(x).unsqueeze(0).unsqueeze(0)  # (vocab,) -> (1, 1, D)  # <3>
        pairs = self._slice_lora(lora_flat) if lora_flat is not None else None
        for i, layer in enumerate(self.layers):
            if pairs is not None:
                q_A, q_B = pairs[i * 2]
                v_A, v_B = pairs[i * 2 + 1]
                x = layer(x, lora_q_A=q_A, lora_q_B=q_B, lora_v_A=v_A, lora_v_B=v_B)
            else:
                x = layer(x)
        return self.head(x.squeeze(0).mean(dim=0))   # (1, 1, D) -> (2,)

1. `_slice_lora` unpacks a flat compiler output vector into `(A, B)` pairs — one pair per LoRA projection, in layer order.
2. `lora_flat=None` (Phase 1): each `LoRALinear` uses its own stored `lora_A`/`lora_B` parameters. `lora_flat` provided (Phase 2): sliced tensors carrying the compiler's `grad_fn` are passed into each projection, so gradients flow back to the compiler without modifying any module state.
3. The BOW input is `(vocab_size,)`. After `self.embed` (a `Linear`), shape is `(D,)`. Two `unsqueeze` calls give `(1, 1, D)` — the `(batch, seq, dim)` shape the transformer expects.

:::{.callout-caution}
The old design used `proj.lora_A = nn.Parameter(A_new)` inside `set_lora`. This creates a new leaf parameter, which **severs the computation graph** to the compiler — `loss.backward()` never reaches the compiler's weights. The functional approach above avoids this by passing the compiler output tensor directly into the forward computation, keeping the full graph intact.

:::

### 5.3 `LoRACompiler`

**Model.** The compiler maps a natural language spec string to all LoRA matrices for the interpreter. The spec encoder is a learned bag-of-words over a small fixed spec vocabulary: each word in the spec is looked up in a learned `nn.Embedding`, the word embeddings are mean-pooled into a single vector, and a 3-layer MLP projects that to the flat LoRA parameter vector.

For our TinyTransformer with $r=4$, $d=64$, 2 layers, 2 LoRA projections per layer (Q, V), each with $A \in \mathbb{R}^{4 \times 64}$ and $B \in \mathbb{R}^{64 \times 4}$, the total output size is $2 \times 2 \times 2 \times (4 \times 64) = 4096$ floats.

In [ ]:
N_TASKS = 3
RANK = 4
D_MODEL = 64
N_TRANSFORMER_LAYERS = 2
N_LORA_PROJ = 2  # Q and V
PER_PROJ = 2 * RANK * D_MODEL  # lora_A + lora_B
LORA_PARAM_COUNT = N_TRANSFORMER_LAYERS * N_LORA_PROJ * PER_PROJ
print(f"Total LoRA parameters: {LORA_PARAM_COUNT}")

# Spec vocabulary: words the compiler can read in a spec string
SPEC_VOCAB = [
    "urgency", "urgent", "billing", "invoice", "payment",
    "meeting", "schedule", "classify", "detect", "email", "is", "if", "about",
]
SPEC_WORD2IDX = {w: i for i, w in enumerate(SPEC_VOCAB)}


class LoRACompiler(nn.Module):
    """Text-spec compiler: spec string -> all LoRA matrices."""
    def __init__(self, spec_vocab_size, spec_embed_dim=32, hidden=128,
                 lora_param_count=LORA_PARAM_COUNT):
        super().__init__()
        self.spec_embed = nn.Embedding(spec_vocab_size, spec_embed_dim)  # <1>
        self.net = nn.Sequential(
            nn.Linear(spec_embed_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Linear(256, lora_param_count),
        )

    def encode_spec(self, spec: str) -> torch.Tensor:
        words = spec.lower().split()
        indices = [SPEC_WORD2IDX[w] for w in words if w in SPEC_WORD2IDX]
        if not indices:
            return torch.zeros(self.spec_embed.embedding_dim)
        ids = torch.tensor(indices)
        return self.spec_embed(ids).mean(dim=0)   # <2>

    def forward(self, spec: str) -> torch.Tensor:   # <3>
        z = self.encode_spec(spec)
        return self.net(z)

1. Each word in `SPEC_VOCAB` gets a learned embedding of dimension `spec_embed_dim`. The embedding table is trained end-to-end in Phase 2.
2. Mean-pooling over the spec words gives a single fixed-size vector regardless of spec length.
3. `forward` takes a raw string — the same API as `compile_spec`. Task IDs are only used for *data generation* (sampling training examples); the compiler itself never sees them.

:::{.callout-note}
This learned BOW encoder generalizes to any spec whose words are in `SPEC_VOCAB`. For production, replace `encode_spec` with a pretrained sentence encoder (e.g. `all-MiniLM-L6-v2`) to handle arbitrary phrasings and unseen vocabulary.

:::

### 5.4 Phase 1: pre-train interpreter

**Training.** We train the TinyTransformer on all 3 tasks jointly. Only the LoRA parameters of each `LoRALinear` are trainable; the base weights and the embedding/head are frozen. We stop phase 1 once accuracy exceeds 85% on a held-out evaluation.

In [ ]:
torch.manual_seed(42)
random.seed(42)

interpreter = TinyTransformer(vocab_size=20, d_model=D_MODEL, n_heads=4, rank=RANK)

# LoRA matrices only
train_params = [p for p in interpreter.parameters() if p.requires_grad]
optimizer_p1 = torch.optim.Adam(train_params, lr=5e-3)
criterion = nn.CrossEntropyLoss()

for step in range(3000):
    task_id = random.randint(0, 2)
    positive = random.random() > 0.5
    x, y = make_sample(task_id, positive)
    logits = interpreter(x)
    loss = criterion(logits.unsqueeze(0), y.unsqueeze(0))
    optimizer_p1.zero_grad()
    loss.backward()
    optimizer_p1.step()
    if (step + 1) % 500 == 0:
        # evaluate on 200 samples
        correct = 0
        with torch.no_grad():
            for _ in range(200):
                tid = random.randint(0, 2)
                xv, yv = make_sample(tid, random.random() > 0.5)
                pred = interpreter(xv).argmax()
                correct += (pred == yv).item()
        acc = correct / 200
        print(f"Step {step+1:>5} | acc: {acc:.3f}")

After phase 1, we freeze all interpreter parameters permanently. The interpreter is now a fixed function $f_{W_0}$ (plus frozen LoRA scaffolding) that the compiler must learn to steer.

In [ ]:
for p in interpreter.parameters():
    p.requires_grad = False

print("Interpreter frozen. Trainable params:",
      sum(p.numel() for p in interpreter.parameters() if p.requires_grad))

### 5.5 Phase 2: train compiler end-to-end

**Training.** The compiler is initialized and trained by generating LoRA parameters for each task, injecting them functionally into the frozen interpreter, running a forward pass, and backpropagating the task loss. Only the compiler's parameters receive gradient updates.

In [ ]:
torch.manual_seed(0)
random.seed(0)

compiler = LoRACompiler(spec_vocab_size=len(SPEC_VOCAB), lora_param_count=LORA_PARAM_COUNT)
optimizer_p2 = torch.optim.Adam(compiler.parameters(), lr=1e-3)

for step in range(3000):
    task_id = random.randint(0, 2)
    spec = TASK_SPECS[task_id]             # <1>
    lora_flat = compiler(spec)             # <2>
    positive = random.random() > 0.5
    x, y = make_sample(task_id, positive)
    logits = interpreter(x, lora_flat=lora_flat)  # <3>
    loss = criterion(logits.unsqueeze(0), y.unsqueeze(0))
    optimizer_p2.zero_grad()
    loss.backward()                        # <4>
    optimizer_p2.step()
    if (step + 1) % 500 == 0:
        print(f"Step {step+1:>5} | loss: {loss.item():.4f}")

1. The spec string (e.g. `"urgency"`) is drawn from `TASK_SPECS` using the sampled task ID. The compiler never sees the integer ID — only the text.
2. `compiler(spec)` encodes the text to a spec embedding and decodes to a flat LoRA vector. The tensor carries a `grad_fn` back into the compiler.
3. `lora_flat` is passed directly into the interpreter's forward — no parameter mutation, graph intact.
4. Gradients flow from the task loss through the interpreter's frozen computation, through the LoRA delta slices, all the way back into the compiler's embedding table and MLP.

**Gradient check.** We verify that compiler parameters actually received gradient updates — a necessary condition for end-to-end training to work. We run one training step and inspect the grad norms of the compiler's spec embedding and MLP output layer:

In [ ]:
torch.manual_seed(1)
random.seed(1)

# Single training step to check gradient flow
_compiler_check = LoRACompiler(spec_vocab_size=len(SPEC_VOCAB), lora_param_count=LORA_PARAM_COUNT)
_opt = torch.optim.Adam(_compiler_check.parameters(), lr=1e-3)

spec = TASK_SPECS[0]  # "urgency"
lora_flat = _compiler_check(spec)
x, y = make_sample(0, positive=True)
logits = interpreter(x, lora_flat=lora_flat)
loss = criterion(logits.unsqueeze(0), y.unsqueeze(0))
loss.backward()

embed_grad_norm = _compiler_check.spec_embed.weight.grad.norm().item()
mlp_out_grad_norm = _compiler_check.net[-1].weight.grad.norm().item()
print(f"spec_embed grad norm:   {embed_grad_norm:.4f}")
print(f"MLP output layer norm:  {mlp_out_grad_norm:.4f}")
assert embed_grad_norm > 0 and mlp_out_grad_norm > 0, "gradient flow broken!"
print("Gradient flow confirmed.")

### 5.6 Evaluation

**Evals.** For each task, we compile exactly once (single forward pass through the compiler), apply the resulting LoRA adapter, and evaluate on 300 held-out samples.

In [ ]:
random.seed(99)

for task_id in range(3):
    spec = TASK_SPECS[task_id]
    with torch.no_grad():
        lora_flat = compiler(spec)

    correct = 0
    n_eval = 300
    with torch.no_grad():
        for _ in range(n_eval):
            positive = random.random() > 0.5
            x, y = make_sample(task_id, positive)
            pred = interpreter(x, lora_flat=lora_flat).argmax()
            correct += (pred == y).item()
    acc = correct / n_eval
    print(f"Task {task_id} ({spec:>10}): accuracy = {acc:.3f}")

The compiler has learned to synthesize a different adapter for each of the three tasks, each accurate at inference time after a single forward pass.

### 5.7 `NeuralFunction` — the agent-ready callable

The evaluation loop above calls `compiler(spec)` and passes `lora_flat` directly into the interpreter's `forward`. We now wrap this into a `NeuralFunction` class and a top-level `compile_spec()` function that hide all the machinery behind a single `__call__`. The result is a Python callable that any agent can use as a tool.

In [ ]:
class NeuralFunction:
    """A compiled neural program: spec -> callable Python function."""

    def __init__(self, spec: str, lora_flat: torch.Tensor, interpreter):
        self.spec = spec
        self._lora_flat = lora_flat   # <1>
        self._interpreter = interpreter

    def __call__(self, x: torch.Tensor) -> int:
        """Run the compiled program on input x; return predicted class label."""
        with torch.no_grad():
            return self._interpreter(x, lora_flat=self._lora_flat).argmax().item()  # <2>

    def __repr__(self):
        return f"NeuralFunction({self.spec!r})"


def compile_spec(spec: str, compiler, interpreter) -> NeuralFunction:   # <3>
    """Compile a natural language spec into a callable neural function."""
    with torch.no_grad():
        lora_flat = compiler(spec)
    return NeuralFunction(spec, lora_flat, interpreter)


# Compile each task into a standalone callable
urgency_fn = compile_spec("urgency", compiler, interpreter)
billing_fn = compile_spec("billing", compiler, interpreter)
meeting_fn = compile_spec("meeting", compiler, interpreter)

print(urgency_fn)
print(billing_fn)
print(meeting_fn)

1. `lora_flat` is generated once by the compiler at construction and stored. All subsequent calls reuse it — no recompilation.
2. Each call is a single frozen forward pass through the interpreter with the cached adapter injected functionally. No state is mutated; concurrent calls on the same interpreter are safe as long as `lora_flat` tensors are distinct.
3. `compile_spec(spec, compiler, interpreter)` is the top-level API — analogous to `paw.compile(...)`. It hides the compiler, the adapter format, and the injection mechanism behind a single Python callable.

Calling the compiled functions on a few examples:

In [ ]:
random.seed(7)

# Sample a positive and negative example for each task
examples = [
    (urgency_fn,  "urgent email",  make_sample(0, positive=True)[0]),
    (urgency_fn,  "neutral email", make_sample(0, positive=False)[0]),
    (billing_fn,  "billing email", make_sample(1, positive=True)[0]),
    (billing_fn,  "neutral email", make_sample(1, positive=False)[0]),
]

print(f"{'Function':<40}  {'Input type':<14}  {'Prediction'}")
print("-" * 65)
for fn, label, x in examples:
    pred = fn(x)
    words = [VOCAB[i] for i, v in enumerate(x) if v > 0]
    print(f"{str(fn):<40}  {label:<14}  {pred}  {words}")

Each `NeuralFunction` is a plain Python callable — it can be passed as a tool to an agent, stored in a dict, or composed with other functions. The compilation cost is paid once at construction; subsequent calls are just a frozen interpreter forward pass.

### 5.8 Agent tool registry

We register compiled functions in a dict and expose a `route` helper — the same pattern an agent framework would use to dispatch tool calls by name:

In [ ]:
# Tool registry: name -> compiled NeuralFunction
TOOLS = {
    "is_urgent": urgency_fn,
    "is_billing": billing_fn,
    "is_meeting": meeting_fn,
}


def run_tool(tool_name: str, x: torch.Tensor) -> int:
    """Dispatch a tool call by name — no recompilation on each call."""
    return TOOLS[tool_name](x)


# Simulate agent dispatching tool calls
random.seed(12)
test_cases = [
    ("is_urgent",  0, True),
    ("is_urgent",  0, False),
    ("is_billing", 1, True),
    ("is_billing", 1, False),
    ("is_meeting", 2, True),
    ("is_meeting", 2, False),
]

print(f"{'Tool':<12}  {'Label':>5}  {'Pred':>5}  {'Correct?'}")
print("-" * 38)
for tool, task_id, pos in test_cases:
    x, y = make_sample(task_id, positive=pos)
    pred = run_tool(tool, x)
    mark = "✓" if pred == y.item() else "✗"
    print(f"{tool:<12}  {y.item():>5}  {pred:>5}  {mark}")

### 5.9 Serialization — saving and reloading a compiled adapter

A `NeuralFunction` is portable: its state is just `spec` (a string) and `lora_flat` (a small tensor). We save these to disk and reconstruct a callable from the artifact — the toy analog of a `.paw` file:

In [ ]:
import os
os.makedirs("tmp", exist_ok=True)

# --- Save ---
artifact = {
    "spec":      urgency_fn.spec,
    "lora_flat": urgency_fn._lora_flat,
    "rank":      RANK,
    "d_model":   D_MODEL,
}
torch.save(artifact, "tmp/compiled_urgency.pt")
print(f"Saved: tmp/compiled_urgency.pt  ({os.path.getsize('tmp/compiled_urgency.pt') / 1024:.1f} KB)")

# --- Reload ---
loaded = torch.load("tmp/compiled_urgency.pt", weights_only=True)
reloaded_fn = NeuralFunction(loaded["spec"], loaded["lora_flat"], interpreter)
print(f"Reloaded: {reloaded_fn}")

# --- Verify identical outputs ---
random.seed(42)
mismatches = 0
for _ in range(200):
    x, _ = make_sample(0, positive=random.random() > 0.5)
    if urgency_fn(x) != reloaded_fn(x):
        mismatches += 1
print(f"Output mismatches over 200 samples: {mismatches}  (expected 0)")

## Section 6: Scaling Up — GPT-2 as Interpreter

### 6.1 Load frozen GPT-2

**Model.** We load the 124M-parameter GPT-2 base model and freeze all parameters. GPT-2 serves as the interpreter: a fixed function whose behavior will be steered entirely by LoRA deltas injected through forward hooks.

In [ ]:
from transformers import GPT2Model, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2 = GPT2Model.from_pretrained("gpt2")
for p in gpt2.parameters():
    p.requires_grad = False

total = sum(p.numel() for p in gpt2.parameters())
print(f"GPT-2 parameters: {total:,}")
print(f"All frozen: {not any(p.requires_grad for p in gpt2.parameters())}")

### 6.2 LoRA injection via `c_attn` hook

GPT-2's attention uses a single `c_attn` Conv1D layer that produces Q, K, V concatenated along the last dimension (output shape: `(batch, seq, 3 * d_model)`). We cannot replace it with `LoRALinear` without rewriting the attention block. Instead, we register a forward hook on the `c_attn` module: the hook intercepts the output and adds LoRA deltas to the Q and V slices, leaving K unmodified.

In [ ]:
def make_qv_lora_hook(lora_B_q, lora_A_q, lora_B_v, lora_A_v, d_model, scale=1.0):
    def hook(module, input, output):
        x = input[0]                                          # <1>
        delta_q = (x @ lora_A_q.T) @ lora_B_q.T              # <2>
        delta_v = (x @ lora_A_v.T) @ lora_B_v.T              # <3>
        out = output.clone()
        out[..., :d_model] += delta_q * scale                 # <4>
        out[..., 2 * d_model: 3 * d_model] += delta_v * scale # <5>
        return out
    return hook

1. `input[0]` is the pre-projection hidden state $x \in \mathbb{R}^{\text{batch} \times \text{seq} \times d}$.
2. Q LoRA delta: $(x A_q^\top) B_q^\top = x \cdot \Delta W_q^\top$.
3. V LoRA delta: $(x A_v^\top) B_v^\top$.
4. The Q slice occupies positions `0:d_model` in the concatenated output.
5. The V slice occupies positions `2*d_model:3*d_model`; K (positions `d_model:2*d_model`) is intentionally left unmodified.

:::{.callout-caution}
K is intentionally left unmodified. Modifying K changes the keys used in attention scoring, which can destabilize the base model's attention patterns. LoRA on Q and V is sufficient to adapt task-relevant representations while keeping attention routing stable.

:::

Registering hooks for one layer to verify the shape:

In [ ]:
GPT2_D_MODEL = 768
LORA_RANK_GPT2 = 4

# Dummy LoRA matrices for layer 0
lora_A_q = torch.randn(LORA_RANK_GPT2, GPT2_D_MODEL) * 0.01
lora_B_q = torch.zeros(GPT2_D_MODEL, LORA_RANK_GPT2)
lora_A_v = torch.randn(LORA_RANK_GPT2, GPT2_D_MODEL) * 0.01
lora_B_v = torch.zeros(GPT2_D_MODEL, LORA_RANK_GPT2)

hook = gpt2.transformer.h[0].attn.c_attn.register_forward_hook(
    make_qv_lora_hook(lora_B_q, lora_A_q, lora_B_v, lora_A_v, GPT2_D_MODEL)
)

tokens = tokenizer("Hello world", return_tensors="pt")["input_ids"]
with torch.no_grad():
    out = gpt2(tokens)
print("Last hidden state shape:", out.last_hidden_state.shape)
hook.remove()

### 6.3 TransformerCompiler

**Model.** The compiler for GPT-2 is a small transformer encoder. It tokenizes the spec string, encodes it to a mean-pooled embedding, and maps that to a flat vector of all LoRA parameters.

For GPT-2 with rank $r = 4$ and $d = 768$, per layer we need:
- $A_q \in \mathbb{R}^{4 \times 768}$, $B_q \in \mathbb{R}^{768 \times 4}$: $2 \times 4 \times 768 = 6144$ params
- $A_v \in \mathbb{R}^{4 \times 768}$, $B_v \in \mathbb{R}^{768 \times 4}$: $6144$ params

Total per layer: $12{,}288$. Across 12 layers: $12 \times 12{,}288 = 147{,}456$ params.

In [ ]:
N_GPT2_LAYERS = 12
PER_LAYER_LORA = 4 * LORA_RANK_GPT2 * GPT2_D_MODEL  # A_q + B_q + A_v + B_v
GPT2_LORA_PARAM_COUNT = N_GPT2_LAYERS * PER_LAYER_LORA
print(f"Total GPT-2 LoRA params: {GPT2_LORA_PARAM_COUNT:,}")


class TransformerCompiler(nn.Module):
    def __init__(self, vocab_size, d_compiler=128, n_heads=4, n_layers=2,
                 lora_param_count=GPT2_LORA_PARAM_COUNT):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_compiler)
        encoder_layer = nn.TransformerEncoderLayer(
            d_compiler, n_heads, dim_feedforward=256, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_compiler, lora_param_count)

    def forward(self, token_ids):
        x = self.embed(token_ids)           # <1>
        x = self.encoder(x)
        return self.head(x.mean(dim=1))     # <2>


gpt2_compiler = TransformerCompiler(vocab_size=tokenizer.vocab_size)
compiler_params = sum(p.numel() for p in gpt2_compiler.parameters())
print(f"Compiler parameters: {compiler_params:,}")

1. Token IDs are embedded into the compiler's $d = 128$ dimensional space.
2. Mean pooling over the sequence dimension produces a single embedding vector for the entire spec string.

### 6.4 Timing and latency profile

We measure compile latency (one GPT-2 compiler forward pass) and call latency (one toy interpreter forward pass), then compare against the estimated LLM API round-trip.

In [ ]:
import time

spec = "Classify if email is urgent"
tokens = tokenizer(spec, return_tensors="pt")["input_ids"]

# warm up
with torch.no_grad():
    _ = gpt2_compiler(tokens)

# --- Compile latency ---
N_COMPILE = 50
t0 = time.perf_counter()
for _ in range(N_COMPILE):
    with torch.no_grad():
        lora_flat_gpt2 = gpt2_compiler(tokens)
t1 = time.perf_counter()
compile_ms = (t1 - t0) / N_COMPILE * 1000

# --- Call latency (toy interpreter for comparison) ---
N_CALLS = 200
x_sample, _ = make_sample(0, positive=True)
# warm up
_ = urgency_fn(x_sample)
t0 = time.perf_counter()
for _ in range(N_CALLS):
    _ = urgency_fn(x_sample)
t1 = time.perf_counter()
call_ms = (t1 - t0) / N_CALLS * 1000

print(f"Compile latency (GPT-2 compiler, {N_COMPILE} runs): {compile_ms:.2f} ms/compile")
print(f"Call latency    (TinyTransformer, {N_CALLS} runs):  {call_ms:.3f} ms/call")
print(f"\nAmortized over 100 calls:")
print(f"  compile_once + 100 * call  = {compile_ms + 100 * call_ms:.1f} ms")
print(f"  compile_each * 100         = {100 * compile_ms:.1f} ms")
print(f"  LLM API call (est.) * 100  = ~{100 * 500:.0f} ms  (500 ms/call typical)")

The compiler runs in a few milliseconds on CPU. A single gradient step of LoRA fine-tuning on GPT-2 — even with a batch size of 1 — requires a full forward and backward pass through the 124M-parameter interpreter, which is orders of magnitude more expensive. Text-to-LoRA amortizes that cost entirely into the (offline) training of the compiler.

**Latency profile: compile-once vs. call-many.** There are two costs to separate:

| Step | What runs | Typical cost |
|------|-----------|-------------|
| `compile(spec)` | One compiler forward pass | ~1–10 ms (CPU) |
| `fn(input)` | One interpreter forward pass | ~1–50 ms (CPU, depends on interpreter size) |
| LLM API call | Network + tokenization + autoregressive decode | 200 ms – 5 s |

The compile step is paid once per task and cached. Subsequent calls skip it entirely — the adapter is already loaded. Each `fn(input)` invocation is a single frozen-model forward pass with no autoregressive decoding and no network round-trip, so it beats a remote LLM API call by one to three orders of magnitude in latency.

**The accuracy–latency trade-off.** The speedup comes with a cost. A compiled `NeuralFunction` can only do what the interpreter model is capable of, constrained further by the low-rank adapter capacity. Tasks that require world knowledge, multi-step reasoning, or handling novel phrasings far from the training distribution will degrade. A frontier LLM called via API handles these gracefully at the cost of latency and per-token fees.

The practical decision rule for agent tool design:

- **Use a compiled function** when the task is well-defined, the input distribution is narrow, latency matters, and the function will be called many times (classification, extraction, format repair, routing).
- **Use an LLM call** when the task requires reasoning, broad knowledge, or flexibility to handle inputs the compiled model has never seen.

:::{.callout-note}
This notebook's GPT-2 compiler is untrained — it will produce random LoRA parameters. The architecture demonstration is complete; full training would require a dataset of (spec, input, output) triples and GPU resources beyond a local CPU run.

:::

## Section 7: Design Considerations

### 7.1 Writing Good Specs

The spec string is the only signal the compiler sees to distinguish tasks. Everything the adapter needs to do must be recoverable from this string alone. The following guidelines apply to any trained text-to-LoRA system.

**Name the decision, not the domain.** The compiler needs to know what output to produce, not just what topic the input is about.

```python
# Bad: describes input domain only
spec = "emails"

# Good: names the required decision
spec = "Classify if email is urgent"
```

**One task per spec.** The adapter targets a fixed-rank delta. Combining tasks splits that capacity and likely does both poorly. Use separate specs and separate adapters.

```python
# Bad: two tasks, one adapter
spec = "Classify urgency and extract billing keywords"

# Good: one task per adapter
spec_a = "Classify if email is urgent"
spec_b = "Classify if email is about billing"
```

**Be explicit about output type.** Binary classification, regression, and extraction all require different adapter behavior. Avoid leaving the output type ambiguous.

```python
# Weak: output type must be inferred
spec = "urgent"

# Better: output type is clear
spec = "Classify if email is urgent"          # binary
spec = "Score urgency of email from 0 to 1"  # regression
```

**Stay close to the training distribution.** The compiler generalizes over the manifold of specs it was trained on. Novel phrasings far from that distribution will not compile reliably, even if semantically equivalent. Treat the spec vocabulary like an API contract — document it and stay within it.

**Avoid negation and conditionals.** Mean-pooled encoders lose word order. A spec like `"Do not classify as urgent unless deadline is mentioned"` will likely have the negation dropped silently.

**Keep specs short.** Mean pooling compresses the spec to a single vector; long specs dilute the signal from task-critical words. If a spec exceeds one sentence, split the task or upgrade to a stronger encoder.

:::{.callout-note}
**Near-boundary behavior.** The compiler interpolates between trained tasks. A spec equidistant between `"billing"` and `"urgency"` in embedding space will produce an adapter that does both weakly. This is expected behavior, not a bug — but it means spec design should keep tasks well-separated in the training vocabulary.

:::

The toy system built here illustrates the general text-to-LoRA skeleton: a compiler encodes a task spec and emits $(B, A)$ matrices that are injected into a frozen interpreter. Scaling this up — replacing the 2-layer encoder with a fine-tuned LLM and the TinyTransformer interpreter with a large base model — leaves the architecture unchanged.

One design question is whether to merge $\Delta W = BA$ into $W_0$ after compilation. The answer is no when the base model is shared across many concurrent adapters. Merging requires a separate copy of the full base model per adapter, scaling memory as $O(n \times |W_0|)$. Keeping adapters unmerged, the footprint is one copy of $W_0$ plus one small $(B, A)$ pair per active task — less than 1% overhead per adapter at rank 4.

## Appendix A1: Rank Selection

The theoretical justification for low-rank updates is the **intrinsic dimensionality** hypothesis: the manifold of useful fine-tuning updates for a pretrained model lies in a much lower-dimensional subspace than the full parameter space. Empirically, ranks between 4 and 16 recover nearly all of the performance of full fine-tuning on most tasks.

We verify this on the toy linear regression from Section 2.4: we fix a target matrix $W^* \in \mathbb{R}^{8 \times 8}$ and train LoRA for ranks $r \in \{1, 2, 4, 8, 16\}$, recording final MSE.

In [ ]:
torch.manual_seed(42)
D = 8
W_star = torch.randn(D, D)

rank_results = {}
for rank in [1, 2, 4, 8, 16]:
    layer = LoRALinear(D, D, rank=rank, alpha=float(rank))
    nn.init.zeros_(layer.linear.weight)
    opt = torch.optim.Adam([layer.lora_A, layer.lora_B], lr=1e-2)
    for _ in range(2000):
        x = torch.randn(32, D)
        loss = ((layer(x) - x @ W_star.T)).pow(2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    rank_results[rank] = loss.item()
    print(f"r={rank:>2} | final MSE: {loss.item():.6f}")

Plotting MSE vs. rank:

In [ ]:
#| code-fold: true
plt.figure(figsize=(5, 3))
plt.plot(list(rank_results.keys()), list(rank_results.values()), marker="o", linewidth=2)
plt.xlabel("Rank $r$")
plt.ylabel("Final MSE")
plt.title("Rank ablation: LoRA toy regression")
plt.xticks(list(rank_results.keys()))
plt.grid(alpha=0.4, linestyle="--")
plt.tight_layout()
plt.show()

**Figure.** Final MSE vs. LoRA rank for the $8 \times 8$ toy regression. Gains diminish sharply after $r = 4$ for this simple task. In practice, the optimal rank depends on task complexity and base model size, but ranks beyond 16 rarely provide significant benefit for fine-tuning large pretrained models.

## Appendix A2: Gradient Flow — Why Functional Injection

The most common mistake when implementing text-to-LoRA is severing the gradient path between the compiler output and the task loss. Two patterns illustrate this:

**Wrong — `.data` mutation breaks the graph.**

```python
# Compiler output
lora_flat = compiler(task_id)  # has grad_fn

# Mutation discards grad_fn:
proj.lora_A.data = lora_flat[:per_mat].view(rank, d_model)
# lora_A is now a Tensor without grad_fn connecting back to compiler
# loss.backward() will not update compiler parameters
```

**Correct — assign new `nn.Parameter` carrying the `grad_fn`.**

```python
A_new = lora_flat[idx: idx + per_mat].view(rank, d_model)  # has grad_fn
proj.lora_A = nn.Parameter(A_new)  # grad_fn preserved
# The forward pass uses A_new, so gradients reach compiler
```

This is the pattern used in the functional `forward` of `TinyTransformer`. For the GPT-2 hook approach, the `lora_A_q` tensor passed into `make_qv_lora_hook` must be the raw compiler output slice (with `grad_fn`), not a detached copy.

A minimal demonstration:

In [ ]:
torch.manual_seed(0)

# Tiny compiler: maps scalar -> 4 weights
tiny_compiler = nn.Linear(1, 4, bias=False)

# Wrong pattern: .data assignment
lora_flat_wrong = tiny_compiler(torch.ones(1, 1))
param_wrong = nn.Parameter(torch.zeros(4))
param_wrong.data = lora_flat_wrong.squeeze()  # severs grad_fn
loss_wrong = (param_wrong.sum() - 10.0).pow(2)
loss_wrong.backward()
print("Wrong pattern - compiler grad:",
      tiny_compiler.weight.grad)  # None or zero

tiny_compiler.zero_grad()

# Correct pattern: Parameter wraps the compiler output directly
lora_flat_correct = tiny_compiler(torch.ones(1, 1))
param_correct = nn.Parameter(lora_flat_correct.squeeze())  # grad_fn intact
loss_correct = (param_correct.sum() - 10.0).pow(2)
loss_correct.backward()
print("Correct pattern - compiler grad:",
      tiny_compiler.weight.grad.flatten())  # non-zero

## Appendix A3: LoRA Merging

Once a LoRA adapter is trained, it can be permanently merged into the base weight:

$$W_\text{merged} = W_0 + s \cdot BA.$$

The merged model is a standard `nn.Linear` with no runtime overhead — no LoRA parameters, no hooks. This is useful for single-adapter deployment where latency is critical.

In [ ]:
def merge_lora(linear: LoRALinear) -> nn.Linear:
    W_merged = linear.linear.weight.data + linear.scale * (linear.lora_B.data @ linear.lora_A.data)
    out_features, in_features = W_merged.shape
    merged = nn.Linear(in_features, out_features, bias=False)
    merged.weight = nn.Parameter(W_merged)
    return merged


# Verify: merged layer should produce the same output as LoRALinear
torch.manual_seed(1)
test_layer = LoRALinear(8, 8, rank=4, alpha=4.0)
merged_layer = merge_lora(test_layer)

x_test = torch.randn(5, 8)
diff = (test_layer(x_test) - merged_layer(x_test)).abs().max().item()
print(f"Max output difference after merging: {diff:.2e}")

The output difference is at floating-point precision.

The shared-base design keeps memory as $O(|W_0| + n \times |\Delta W|)$ rather than $O(n \times |W_0|)$: one frozen base model serves all adapters concurrently, with each task identified by its own $(B, A)$ pair. At rank 4, the LoRA overhead per adapter is less than 1% of the base model size.

---

■